# Edge-IIoTset Dataset Audit & Leakage Analysis

This notebook imports the `src.audit` functions to analyze the dataset, display distributions, and report on leakage-prone features.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

from src.config import AUDIT_DIR, FIGURES_DIR, DEFAULT_TARGET_COL
from src.data_loader import get_csv_files, load_dataset_sample

# Setup plot style
sns.set_theme(style="whitegrid")

## 1. Run Audit Pipeline

In [ ]:
from src.audit import run_audit

# This will scan the dataset, create feature policies, and dump them to results/audit
run_audit()

## 2. Review Dataset Metadata

In [ ]:
with open(AUDIT_DIR / "dataset_summary.json", "r") as f:
    metadata = json.load(f)

print(f"File: {metadata.get('filename')}")
print(f"Rows: {metadata.get('num_rows')}")
print(f"Columns: {metadata.get('num_columns')}")

## 3. Visualize Class Distribution

In [ ]:
class_dist = pd.read_csv(AUDIT_DIR / "class_distribution.csv")

plt.figure(figsize=(8, 5))
sns.barplot(data=class_dist, x='class', y='count', palette='viridis')
plt.title(f'Class Distribution ({DEFAULT_TARGET_COL})')
plt.xlabel('Class Label')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_distribution.png")
plt.show()

## 4. Review Leakage Policies

In [ ]:
audit_df = pd.read_csv(AUDIT_DIR / "feature_audit.csv")
display(audit_df.groupby('category').size())

print("\nFeatures dropped due to leakage policy:")
print(audit_df[audit_df['recommended_action'].str.contains('Drop')]['feature'].tolist())